In [ ]:

# TITANIC DATA PREPROCESSING - COMPLETE DAILY CHALLENGE

# IMPORT LIBRARIES


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    LabelEncoder,
    OneHotEncoder,
    StandardScaler,
    MinMaxScaler
)


# LOAD DATASET


df = pd.read_csv("train.csv")

print("First 5 rows:")
print(df.head())

print("\nDataset Shape:")
print(df.shape)



# EXERCISE 1 - DUPLICATE DETECTION & REMOVAL


print("\n================ EXERCISE 1 ================\n")

# Count duplicates
duplicates = df.duplicated().sum()

print(f"Number of duplicate rows: {duplicates}")

# Shape before removal
before_rows = df.shape[0]

# Remove duplicates
df = df.drop_duplicates()

# Shape after removal
after_rows = df.shape[0]

print(f"Rows before removing duplicates: {before_rows}")
print(f"Rows after removing duplicates: {after_rows}")










In [ ]:
# EXERCISE 2 - HANDLING MISSING VALUES

print("\n================ EXERCISE 2 ================\n")

# Check missing values
print(df.isnull().sum())


# AGE COLUMN
# Use median because Age contains numerical values
# and may contain outliers


age_imputer = SimpleImputer(strategy="median")
df["Age"] = age_imputer.fit_transform(df[["Age"]])

# EMBARKED COLUMN
# Use most frequent value


embarked_imputer = SimpleImputer(strategy="most_frequent")
df["Embarked"] = embarked_imputer.fit_transform(df[["Embarked"]]).ravel()


# CABIN COLUMN
# Too many missing values
# Replace with constant value


df["Cabin"] = df["Cabin"].fillna("Unknown")

print("\nMissing values after treatment:")
print(df.isnull().sum())


In [ ]:
# EXERCISE 3 - FEATURE ENGINEERING


print("\n================ EXERCISE 3 ================\n")


# CREATE FAMILY SIZE


df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

# EXTRACT TITLE FROM NAME

df["Title"] = df["Name"].str.extract(r',\s*([^\.]*)\s*\.')

print(df[["Name", "Title"]].head())


# LABEL ENCODING TITLE


label_encoder = LabelEncoder()

df["TitleEncoded"] = label_encoder.fit_transform(df["Title"])

print("\nEncoded Titles:")
print(df[["Title", "TitleEncoded"]].head())

In [ ]:
# EXERCISE 4 - OUTLIER DETECTION & TREATMENT


print("\n================ EXERCISE 4 ================\n")


# VISUALIZATION BEFORE


plt.figure(figsize=(8, 4))
plt.boxplot(df["Fare"])
plt.title("Fare Before Outlier Treatment")
plt.show()


# IQR METHOD


Q1 = df["Fare"].quantile(0.25)
Q3 = df["Fare"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

# CAPPING OUTLIERS


upper_limit = df["Fare"].quantile(0.98)

df["Fare"] = np.where(
    df["Fare"] > upper_limit,
    upper_limit,
    df["Fare"]
)


# VISUALIZATION AFTER


plt.figure(figsize=(8, 4))
plt.boxplot(df["Fare"])
plt.title("Fare After Outlier Treatment")
plt.show()

In [ ]:
# EXERCISE 5 - STANDARDIZATION & NORMALIZATION


print("\n================ EXERCISE 5 ================\n")

# STANDARDIZATION
# For approximately normal distributions


standard_scaler = StandardScaler()

df["Age_Standardized"] = standard_scaler.fit_transform(df[["Age"]])


# NORMALIZATION
# For skewed/bounded values


minmax_scaler = MinMaxScaler()

df["Fare_Normalized"] = minmax_scaler.fit_transform(df[["Fare"]])

print(df[[
    "Age",
    "Age_Standardized",
    "Fare",
    "Fare_Normalized"
]].head())



# EXERCISE 6 - ENCODING CATEGORICAL FEATURES


print("\n================ EXERCISE 6 ================\n")


# IDENTIFY CATEGORICAL COLUMNS


categorical_columns = ["Sex", "Embarked", "Title"]


# ONE-HOT ENCODING


df_encoded = pd.get_dummies(
    df,
    columns=categorical_columns,
    drop_first=True
)

print("\nDataset after encoding:")
print(df_encoded.head())

In [ ]:
# EXERCISE 6 - ENCODING CATEGORICAL FEATURES


print("\n================ EXERCISE 6 ================\n")


# IDENTIFY CATEGORICAL COLUMNS


categorical_columns = ["Sex", "Embarked", "Title"]


# ONE-HOT ENCODING


df_encoded = pd.get_dummies(
    df,
    columns=categorical_columns,
    drop_first=True
)

print("\nDataset after encoding:")
print(df_encoded.head())


In [ ]:
# EXERCISE 7 - AGE GROUP TRANSFORMATION


print("\n================ EXERCISE 7 ================\n")

# CREATE AGE GROUPS


bins = [0, 12, 18, 60, 100]

labels = ["Child", "Teenager", "Adult", "Senior"]

df_encoded["AgeGroup"] = pd.cut(
    df_encoded["Age"],
    bins=bins,
    labels=labels
)

print(df_encoded[["Age", "AgeGroup"]].head())


# ONE-HOT ENCODE AGE GROUPS


df_encoded = pd.get_dummies(
    df_encoded,
    columns=["AgeGroup"]
)

print("\nDataset after AgeGroup encoding:")
print(df_encoded.head())



# FINAL DATASET INFO


print("\n================ FINAL DATASET ================\n")

print(df_encoded.info())

print("\nFinal Shape:")
print(df_encoded.shape)